In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.etl.context import PipelineContext
from src.etl.run_tracker import PipelineRun
from src.etl import extract

13:09:20 | INFO    | etl          | Logging to: C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\logs\etl_20260511_130920.log


In [2]:
import duckdb
from pathlib import Path

parquet_path = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim\cicids_clean.parquet")

schema = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{parquet_path.as_posix()}')").df()

# Check event_time specifically
print("event_time column type:")
print(schema[schema['column_name'] == 'event_time'])

# Also check is_attack and source_day types
print("\nLabel-related types:")
print(schema[schema['column_name'].isin(['label_clean', 'attack_family', 'is_attack', 'source_day'])])

event_time column type:
   column_name column_type null   key default extra
86  event_time   TIMESTAMP  YES  None    None  None

Label-related types:
      column_name column_type null   key default extra
82    label_clean     VARCHAR  YES  None    None  None
83  attack_family     VARCHAR  YES  None    None  None
84      is_attack     INTEGER  YES  None    None  None
85     source_day     VARCHAR  YES  None    None  None


In [3]:
print("=" * 60)
print("Test 1: Sample mode (small, fast)")
print("=" * 60)

ctx = PipelineContext(run_mode="sample", sample_size=10_000)

with PipelineRun(ctx) as run:
    with run.stage("extract") as stage:
        df = extract.run(ctx)
        stage.set_metrics(
            rows_in=0,
            rows_out=ctx.rows_extracted,
            notes=f"sample mode, requested={ctx.sample_size}"
        )

print(f"\n✅ Extracted {ctx.rows_extracted:,} rows")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst 3 rows (key columns only):")
print(df[['event_time', 'Source IP', 'Destination IP', 'attack_family', 'label_clean']].head(3))

Test 1: Sample mode (small, fast)
13:09:22 | INFO    | etl.tracker  | ╔══ Pipeline run 6 started: cicids_to_warehouse (sample) ══╗
13:09:22 | INFO    | etl.tracker  | ▶ Starting stage: extract
13:09:22 | INFO    | etl.extract  | Reading stratified sample (n=10,000) from cicids_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:09:39 | INFO    | etl.extract  | Loaded 8,047 rows × 87 columns into memory
13:09:39 | WARNING | etl.extract  | Dropped 1,000 invalid rows (missing timestamp or IPs)
13:09:39 | INFO    | etl.extract  | Attack family distribution (top): {'DDoS': 1000, 'DoS': 1000, 'Botnet': 1000, 'Brute Force': 1000, 'Benign': 1000, 'Web Attack': 1000, 'Reconnaissance': 1000, 'Infiltration': 36, 'Exploit': 11}
13:09:39 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 17.1s (rows: 0 → 7,047)
13:09:39 | INFO    | etl.tracker  | ╚══ Pipeline run 6 SUCCESS in 17.5s ══╝

✅ Extracted 7,047 rows

DataFrame shape: (7047, 87)

First 3 rows (key columns only):
           event_time   Source IP Destination IP attack_family label_clean
0 2017-07-07 16:15:00  172.16.0.1  192.168.10.50          DDoS        DDoS
1 2017-07-07 16:03:00  172.16.0.1  192.168.10.50          DDoS        DDoS
2 2017-07-07 16:01:00  172.16.0.1  192.168.10.50          DDoS        DDoS


In [4]:
print("=" * 60)
print("Test 2: Full mode (entire dataset)")
print("=" * 60)

ctx = PipelineContext(run_mode="full")

with PipelineRun(ctx) as run:
    with run.stage("extract") as stage:
        df = extract.run(ctx)
        stage.set_metrics(
            rows_in=0,
            rows_out=ctx.rows_extracted,
            notes="full mode"
        )

print(f"\n✅ Extracted {ctx.rows_extracted:,} rows")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nDate range:")
print(f"  Earliest: {df['event_time'].min()}")
print(f"  Latest:   {df['event_time'].max()}")
print(f"\nClass distribution:")
print(df['attack_family'].value_counts())

Test 2: Full mode (entire dataset)
13:10:15 | INFO    | etl.tracker  | ╔══ Pipeline run 7 started: cicids_to_warehouse (full) ══╗
13:10:15 | INFO    | etl.tracker  | ▶ Starting stage: extract
13:10:15 | INFO    | etl.extract  | Reading cicids_clean.parquet (293.0 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:10:44 | INFO    | etl.extract  | Loaded 3,119,345 rows × 87 columns into memory
13:10:59 | WARNING | etl.extract  | Dropped 288,602 invalid rows (missing timestamp or IPs)
13:11:01 | INFO    | etl.extract  | Attack family distribution (top): {'Benign': 2273097, 'DoS': 252661, 'Reconnaissance': 158930, 'DDoS': 128027, 'Brute Force': 13835, 'Web Attack': 2180, 'Botnet': 1966, 'Infiltration': 36, 'Exploit': 11}
13:11:02 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 47.0s (rows: 0 → 2,830,743)
13:11:02 | INFO    | etl.tracker  | ╚══ Pipeline run 7 SUCCESS in 47.1s ══╝

✅ Extracted 2,830,743 rows

Memory usage: 2920.2 MB

Date range:
  Earliest: 2017-07-03 08:55:58
  Latest:   2017-07-07 17:02:00

Class distribution:
attack_family
Benign            2273097
DoS                252661
Reconnaissance     158930
DDoS               128027
Brute Force         13835
Web Attack           2180
Botnet               1966
Infiltration           36
Exploit                11
Name: count, dt

In [5]:
print("=" * 60)
print("Test 3: Failure handling (bad path)")
print("=" * 60)

ctx = PipelineContext(run_mode="sample", sample_size=100)
ctx.input_parquet = Path("does_not_exist.parquet")

try:
    with PipelineRun(ctx) as run:
        with run.stage("extract") as stage:
            df = extract.run(ctx)
            stage.set_metrics(rows_out=0)
    print("⚠️ Expected exception was not raised")
except FileNotFoundError as e:
    print(f"✅ Caught expected error: {str(e)[:80]}...")
except Exception as e:
    print(f"⚠️ Unexpected exception type {type(e).__name__}: {e}")

Test 3: Failure handling (bad path)
13:11:59 | INFO    | etl.tracker  | ╔══ Pipeline run 8 started: cicids_to_warehouse (sample) ══╗
13:11:59 | INFO    | etl.tracker  | ▶ Starting stage: extract
13:11:59 | ERROR   | etl.tracker  | ✗ Stage 'extract' failed after 0.0s: Cleaned CICIDS parquet not found at: does_not_exist.parquet
Run notebooks/01_cicids2017_eda.ipynb to generate it.
13:11:59 | ERROR   | etl.tracker  | ╚══ Pipeline run 8 FAILED: Cleaned CICIDS parquet not found at: does_not_exist.parquet
Run notebooks/01_cicids2017_eda.ipynb to generate it. ══╝
✅ Caught expected error: Cleaned CICIDS parquet not found at: does_not_exist.parquet
Run notebooks/01_cic...


In [6]:
import pandas as pd
from src.warehouse import get_engine

print("=" * 60)
print("Test 4: Meta table review")
print("=" * 60)

engine = get_engine()

print("\nRecent pipeline runs:")
runs_df = pd.read_sql("""
    SELECT run_id, run_mode, status, started_at, duration_seconds,
           rows_extracted, error_message
    FROM warehouse.meta_pipeline_runs
    ORDER BY run_id DESC
    LIMIT 5
""", engine)
print(runs_df.to_string(index=False))

print("\nRecent stages:")
stages_df = pd.read_sql("""
    SELECT stage_id, run_id, stage_name, status,
           duration_seconds, rows_in, rows_out, notes
    FROM warehouse.meta_pipeline_stages
    ORDER BY stage_id DESC
    LIMIT 10
""", engine)
print(stages_df.to_string(index=False))

Test 4: Meta table review

Recent pipeline runs:
 run_id run_mode  status                 started_at  duration_seconds  rows_extracted                                                                                                             error_message
      8   sample  failed 2026-05-11 13:11:59.538709              0.02               0        Cleaned CICIDS parquet not found at: does_not_exist.parquet\nRun notebooks/01_cicids2017_eda.ipynb to generate it.
      7     full success 2026-05-11 13:10:15.016226             47.06         2830743                                                                                                                      None
      6   sample success 2026-05-11 13:09:21.857742             17.47            7047                                                                                                                      None
      5   sample  failed 2026-05-11 13:08:19.187565             27.50               0 Parquet is missing required colum